# 16 - QAOA for MaxCut

Solve MaxCut on a small graph using the Quantum Approximate Optimization Algorithm.

**Concepts:** Combinatorial optimization, cost/mixer layers, variational circuit

In [ ]:
import quantsdk as qs
import math
import numpy as np

## Triangle Graph

Edges: (0,1), (1,2), (0,2). MaxCut = 2.

In [ ]:
edges = [(0, 1), (1, 2), (0, 2)]
n = 3

def qaoa_circuit(gamma: float, beta: float) -> qs.Circuit:
    """QAOA p=1 circuit for MaxCut on a triangle."""
    circuit = qs.Circuit(n, name="QAOA")
    
    # Initial superposition
    for i in range(n):
        circuit.h(i)
    
    # Cost layer: e^(-i*gamma*C)
    for i, j in edges:
        circuit.cx(i, j)
        circuit.rz(j, 2 * gamma)
        circuit.cx(i, j)
    
    # Mixer layer: e^(-i*beta*B)
    for i in range(n):
        circuit.rx(i, 2 * beta)
    
    circuit.measure_all()
    return circuit

def maxcut_value(bitstring: str) -> int:
    """Count edges crossing the cut."""
    return sum(1 for i, j in edges if bitstring[i] != bitstring[j])

In [ ]:
def expected_cut(gamma: float, beta: float, shots: int = 4000) -> float:
    """Compute expected MaxCut value."""
    circuit = qaoa_circuit(gamma, beta)
    result = qs.run(circuit, shots=shots, seed=42)
    total = sum(maxcut_value(bs) * count for bs, count in result.counts.items())
    return total / shots

# Grid search
best_cut = 0
best_params = (0, 0)
gammas = np.linspace(0, math.pi, 15)
betas = np.linspace(0, math.pi / 2, 15)

for g in gammas:
    for b in betas:
        cut = expected_cut(g, b)
        if cut > best_cut:
            best_cut = cut
            best_params = (g, b)

print(f"Best gamma={best_params[0]:.3f}, beta={best_params[1]:.3f}")
print(f"Expected cut: {best_cut:.3f} (optimal=2)")

In [ ]:
# Run with best parameters
circuit = qaoa_circuit(*best_params)
result = qs.run(circuit, shots=2000, seed=42)

print("QAOA results:")
for bs, count in sorted(result.counts.items(), key=lambda x: -x[1]):
    cut = maxcut_value(bs)
    print(f"  |{bs}>: count={count}, cut={cut} {'<-- optimal' if cut == 2 else ''}")